In [16]:
# %%
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 1 — Carga y Optimización de Multi-Bases de Datos   ║
# ╚══════════════════════════════════════════════════════════╝
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# === 1. CONFIGURACIÓN DE RUTAS Y CONSTANTES ===
PATH_INDICES = 'csv_dashboard/BaseINDICES-2020-2025.csv'
PATH_ING_CHILE = 'csv_dashboard/todas_las_ingenierias_chile.csv'

COL_INST_INDICES = 'Nombre Institución'
COL_REG_INDICES = 'Nombre Region'
COL_CARRERA_INDICES = 'Carrera Genérica'

# === 2. CARGA Y LIMPIEZA: BASE HISTÓRICA INDICES (2020-2025) ===
try:
    df_indices = pd.read_csv(PATH_INDICES, sep=';', encoding='utf-8')
except Exception:
    df_indices = pd.read_csv(PATH_INDICES, sep=',', encoding='utf-8')

cols_num_indices = ['Matrícula primer año hombres', 'Matrícula primer año mujeres', 'Vacantes', 'Matrícula Primer Año', 'Valor de arancel', 'Matrícula Total']
for col in cols_num_indices:
    if col in df_indices.columns:
        if df_indices[col].dtype == 'object':
            df_indices[col] = df_indices[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df_indices[col] = pd.to_numeric(df_indices[col], errors='coerce').astype('float32')

df_indices['Año'] = pd.to_numeric(df_indices['Año'], errors='coerce').fillna(2024).astype('int16')
df_ing_indices = df_indices[df_indices[COL_CARRERA_INDICES].str.contains('Ingeniería', case=False, na=False)].copy()

# === 3. CARGA Y LIMPIEZA: BASE NACIONAL (todas_las_ingenierias_chile.csv) ===
df_nacional = None

# Probamos los encodings más comunes en los que Excel exporta en Windows
for encoding_test in ['utf-16', 'latin-1', 'utf-8', 'cp1252']:
    try:
        # Intentamos primero con punto y coma (;) que es tu separador oficial
        df_nacional = pd.read_csv(PATH_ING_CHILE, sep=';', encoding=encoding_test)
        print(f"-> Base nacional cargada con éxito usando encoding: '{encoding_test}' y separador ';'")
        break
    except Exception:
        try:
            # Si falla por el separador, intentamos con coma (,) por si acaso
            df_nacional = pd.read_csv(PATH_ING_CHILE, sep=',', encoding=encoding_test)
            print(f"-> Base nacional cargada con éxito usando encoding: '{encoding_test}' y separador ','")
            break
        except Exception:
            continue

if df_nacional is None:
    raise ValueError("❌ No se pudo decodificar el archivo 'todas_las_ingenierias_chile.csv'. Revisa el formato.")
# Limpieza estricta de strings con %, puntos o caracteres raros en la nueva base
cols_num_nacional = [
    '% Titulados continuidad de estudios', 'Retención de 1er año', 
    'Duración Real (semestres)', 'Empleabilidad al 1er año', 
    'Empleabilidad al 2º Año', 'Ingreso promedio al 4° año'
]

for col in cols_num_nacional:
    if col in df_nacional.columns:
        if df_nacional[col].dtype == 'object':
            # Quitamos el signo %, los puntos de miles y cambiamos comas por puntos decimales
            df_nacional[col] = df_nacional[col].astype(str).str.replace('%', '', regex=False)
            df_nacional[col] = df_nacional[col].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df_nacional[col] = pd.to_numeric(df_nacional[col], errors='coerce').astype('float32')

# === 4. 🔥 SISTEMA DE CACHING INDEXADO (Compilando todo en RAM) ===
print("Pre-calculando tablas caché de alto rendimiento...")

# Caché indices: Para tendencias y mapas
cache_graficos = df_ing_indices.groupby(['Año', COL_CARRERA_INDICES])[['Matrícula primer año hombres', 'Matrícula primer año mujeres', 'Matrícula Total', 'Valor de arancel']].agg({
    'Matrícula primer año hombres': 'sum', 'Matrícula primer año mujeres': 'sum', 'Matrícula Total': 'sum', 'Valor de arancel': 'median'
}).reset_index()

cache_mapas = df_ing_indices.groupby(['Año', COL_REG_INDICES])[['Matrícula Total', 'Matrícula Primer Año', 'Valor de arancel']].agg({
    'Matrícula Total': 'sum', 'Matrícula Primer Año': 'sum', 'Valor de arancel': 'median'
}).reset_index()

# Caché Nacional: Extraemos directo los datos reales de Empleabilidad, Retención e Ingresos
cache_kpi_real = df_nacional.copy()

print(f"✅ DATA STORE CONTROL: Bases sincronizadas.")
print(f"-> Histórico: {len(df_ing_indices)} registros | Nacional: {len(cache_kpi_real)} registros.")

-> Base nacional cargada con éxito usando encoding: 'utf-16' y separador ';'
Pre-calculando tablas caché de alto rendimiento...
✅ DATA STORE CONTROL: Bases sincronizadas.
-> Histórico: 75 registros | Nacional: 306 registros.


In [17]:
# %%
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 2 — Diseño de la Interfaz Visual y Contenedores    ║
# ╚══════════════════════════════════════════════════════════╝
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import plotly.express as px

# 1. Inyección de estilos CSS para forzar bordes redondos en los botones
estilos_css = widgets.HTML("""
<style>
    .tabs-redondeadas .btn {
        border-radius: 16px !important; 
        margin-right: 6px !important;   
        border: 1px solid #bce8f1 !important;
    }
</style>
""")

# 2. Barra de Navegación Superior
tabs_navegacion = widgets.ToggleButtons(
    options=['KPIs', 'GRÁFICOS', 'MAPAS', 'ML (RANDOM FOREST PESOS)'],
    value='KPIs',
    button_style='info',
    layout=widgets.Layout(width='100%', margin='0px 0px 5px 0px')
)
tabs_navegacion.add_class('tabs-redondeadas') 

linea_separadora = widgets.HTML("<hr style='border: 0; border-top: 3px solid #000000; margin: 5px 0px 15px 0px; width: 100%; opacity: 1;'>")


# =====================================================================
# MAQUETACIÓN INTERNA: PESTAÑA 'KPIs' (DOS SELECTORES IZQ / TARJETAS DER)
# =====================================================================
selector_kpi_institucion = widgets.Dropdown(
    options=['---', 'Universidad de Chile', 'Pontificia Universidad Catolica', 'Universidad de Concepcion', 'Universidad Austral de Chile'],
    value='---',
    description='Institucion:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

selector_kpi_carrera = widgets.Dropdown(
    options=['---', 'Ingenieria Civil Informatica', 'Ingenieria Civil Industrial', 'Ingenieria Civil en Obras Civiles', 'Ingenieria Comercial'],
    value='---',
    description='Carrera:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='5px 0px 15px 0px')
)

panel_izquierdo_kpis = widgets.VBox([
    widgets.HTML("<h4>Filtros KPI</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_kpi_institucion,
    selector_kpi_carrera,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>"
                 "<i>Selecciona una institucion y una carrera especifica para cargar los indicadores clave de rendimiento asociados.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_kpi_cards = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_kpis = widgets.HBox([panel_izquierdo_kpis, area_kpi_cards], layout=widgets.Layout(width='100%', height='100%', padding='5px'))


# =====================================================================
# MAQUETACIÓN INTERNA: PESTAÑA 'GRÁFICOS'
# =====================================================================
diccionario_graficos = {
    '---': '',
    'Brecha de Genero en Matematicas (M1 vs M2)': 'csv/boxplot_genero_m1m2.png',
    'Distribucion por Rama Educacional (HC vs TP)': 'csv/violin_rama_educacional.png',
    'Impacto del Programa PACE': 'csv/bar_pace_impact.png',
    'Relacion Competencia Lectora vs Matematicas': 'csv/jointplot_lect_mate.png',
    'Evolucion del Puntaje por Dependencia': 'csv/lineplot_gap_evolution.png'
}

selector_graficos = widgets.Dropdown(
    options=list(diccionario_graficos.keys()),
    value='---',
    description='Grafico:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

selector_dimensiones = widgets.Dropdown(
    options=['---', 'Genero', 'Dependencia', 'Rama Educacional', 'PACE'],
    value='---',
    description='Dimensiones:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

panel_izquierdo_graficos = widgets.VBox([
    widgets.HTML("<h4>Reportes Estadisticos</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_graficos,
    selector_dimensiones,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>"
                 "<i>Selecciona un reporte estadistico y su dimension para cargar el analisis exploratorio de datos.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_imagen_grafico = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_graficos = widgets.HBox([panel_izquierdo_graficos, area_imagen_grafico], layout=widgets.Layout(width='100%', height='100%', padding='5px'))


# =====================================================================
# MAQUETACIÓN INTERNA: PESTAÑA 'MAPAS'
# =====================================================================
opciones_mapas = [
    '---',
    '🗺 Empleabilidad al 1er Año',
    '🗺 Retención de Primer Año',
    '🗺 Mejor Sueldo al 4to Año'
]

selector_mapas = widgets.Dropdown(
    options=opciones_mapas,
    value='---',
    description='Ver mapa:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

panel_izquierdo_mapas = widgets.VBox([
    widgets.HTML("<h4>Variables del Mapa</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_mapas,
    widgets.HTML("""
        <div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>
            <b>Cómo usar:</b><br>
            🖱 Hover sobre una región para ver detalles<br>
            🏆 Muestra la carrera líder de cada región<br><br>
            <span style='color:#aaa; font-size:0.9em;'>Fuente: SIES / Mifuturo</span>
        </div>
    """)
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_imagen_mapa = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_mapas = widgets.HBox([panel_izquierdo_mapas, area_imagen_mapa], layout=widgets.Layout(width='100%', height='100%', padding='5px'))


# =====================================================================
# OTRAS PESTAÑAS (LIENZOS LIMPIOS COMPLETOS)
# =====================================================================
layout_tab_ml = widgets.VBox([
    widgets.HTML("<div style='padding: 40px; text-align: center; color: #999; font-style: italic; font-family: sans-serif;'>"
                 "<h3>[ Pestaña de ML Limpia ]</h3>Espacio disponible para analisis predictivo.</div>")
], layout=widgets.Layout(width='100%', height='100%'))


# Contenedor dinamico central para el intercambio de vistas
contenedor_cuerpo = widgets.Output(layout=widgets.Layout(width='100%', height='430px', overflow='auto'))

print("Celda 2: Arquitectura visual de la interfaz cargada en la memoria.")

Celda 2: Arquitectura visual de la interfaz cargada en la memoria.


In [18]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELDA MAPAS vFINAL — Coropletas Dinámicas de Chile                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

import geopandas as gpd
import plotly.graph_objects as go
import json, unicodedata
import numpy as np
import pandas as pd
from shapely.geometry import box as _box
from IPython.display import HTML

# ── PASO 1: CONVERTIR SUELDOS ──────────────────────────────────────────
def _sueldo_a_num(texto):
    if pd.isna(texto) or str(texto).strip().lower() == 's/i':
        return np.nan
    t = str(texto).lower()
    if 'sobre' in t and '3' in t and '500' in t: return 3750000
    if '3 millones 500' in t: return 3750000
    if '3 millones a' in t: return 3250000
    if '3 millones' in t: return 3250000
    if '2 millones 500' in t: return 2750000
    if '2 millones 400' in t: return 2450000
    if '2 millones 300' in t: return 2350000
    if '2 millones 200' in t: return 2250000
    if '2 millones 100' in t: return 2150000
    if '2 millones' in t: return 2050000
    for val, num in [('900',1950000),('800',1850000),('700',1750000),
                     ('600',1650000),('500',1550000),('400',1450000),
                     ('300',1350000),('200',1250000),('100',1150000)]:
        if '1 mill' in t and val in t: return num
    if '1 mill' in t: return 1050000
    return np.nan

# ── PASO 2: SHAPEFILE ─────────────────────────────────────────────────
print("Cargando shapefile...")
_gdf_raw = gpd.read_file('../regiones/Regional.shp').to_crs(epsg=4326)

_col_reg = None
for _c in ['NOM_REG', 'NOM_REGION', 'REGION', 'nom_reg', 'Region', 'NOMBRE', 'NAME']:
    if _c in _gdf_raw.columns:
        _col_reg = _c
        break
if _col_reg is None:
    _col_reg = [c for c in _gdf_raw.columns if _gdf_raw[c].dtype == object][0]
print(f"  Columna región: '{_col_reg}'")

# Recortar bbox para excluir Rapa Nui y Antártica
_bbox_chile = _box(-76.0, -56.0, -64.0, -17.0)
_gdf_raw = _gdf_raw.copy()
_gdf_raw['geometry'] = _gdf_raw.geometry.make_valid()
_gdf_raw['geometry'] = _gdf_raw['geometry'].intersection(_bbox_chile)
_gdf_raw = _gdf_raw[~_gdf_raw['geometry'].is_empty].reset_index(drop=True)
print(f"  Regiones tras filtrar bbox: {len(_gdf_raw)}")

# ── PASO 3: NORMALIZACIÓN DE CLAVES ───────────────────────────────────
def _norm(s):
    s = unicodedata.normalize('NFKD', str(s).strip().lower()).encode('ascii','ignore').decode()
    s = s.replace('-', '').replace('.', '').replace("'", '').replace('  ', ' ').strip()
    return s

# ── PASO 4: DICCIONARIO INSTITUCIÓN → REGIÓN ──────────────────────────
_INST_A_REGION = {
    'UNIVERSIDAD DE CHILE': 'Región Metropolitana de Santiago',
    'PONTIFICIA UNIVERSIDAD CATÓLICA DE CHILE': 'Región Metropolitana de Santiago',
    'PONTIFICIA UNIVERSIDAD CATOLICA DE CHILE': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD DE SANTIAGO DE CHILE': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD DIEGO PORTALES': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD ANDRES BELLO': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD ADOLFO IBAÑEZ': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD FINIS TERRAE': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD MAYOR': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD CENTRAL DE CHILE': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD SANTO TOMAS': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD DE LAS AMERICAS': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD ALBERTO HURTADO': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD DEL DESARROLLO': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD AUTONOMA DE CHILE': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD SAN SEBASTIAN': 'Región Metropolitana de Santiago',
    "UNIVERSIDAD BERNARDO O'HIGGINS": 'Región Metropolitana de Santiago',
    'UNIVERSIDAD BERNARDO O´HIGGINS': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD TECNOLOGICA DE CHILE INACAP': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD DE LOS ANDES': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD CATOLICA CARDENAL RAUL SILVA HENRIQUEZ': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD TECNOLOGICA METROPOLITANA': 'Región Metropolitana de Santiago',
    'UNIVERSIDAD DE ARTES, CIENCIAS Y COMUNICACION - UNIACC': 'Región Metropolitana de Santiago',
    'PONTIFICIA UNIVERSIDAD CATÓLICA DE VALPARAÍSO': 'Región de Valparaíso',
    'PONTIFICIA UNIVERSIDAD CATOLICA DE VALPARAISO': 'Región de Valparaíso',
    'UNIVERSIDAD TÉCNICA FEDERICO SANTA MARÍA': 'Región de Valparaíso',
    'UNIVERSIDAD TECNICA FEDERICO SANTA MARIA': 'Región de Valparaíso',
    'UNIVERSIDAD DE VALPARAISO': 'Región de Valparaíso',
    'UNIVERSIDAD DE VIÑA DEL MAR': 'Región de Valparaíso',
    'UNIVERSIDAD DE PLAYA ANCHA DE CIENCIAS DE LA EDUCACION': 'Región de Valparaíso',
    'UNIVERSIDAD DE CONCEPCION': 'Región del Biobío',
    'UNIVERSIDAD DEL BIO-BIO': 'Región del Biobío',
    'UNIVERSIDAD CATOLICA DE LA SANTISIMA CONCEPCION': 'Región del Biobío',
    'UNIVERSIDAD AUSTRAL DE CHILE': 'Región de Los Ríos',
    'UNIVERSIDAD DE LOS LAGOS': 'Región de Los Lagos',
    'UNIVERSIDAD DE LA FRONTERA': 'Región de La Araucanía',
    'UNIVERSIDAD CATOLICA DE TEMUCO': 'Región de La Araucanía',
    'UNIVERSIDAD ADVENTISTA DE CHILE': 'Región de La Araucanía',
    'UNIVERSIDAD DE TALCA': 'Región del Maule',
    'UNIVERSIDAD CATÓLICA DEL MAULE': 'Región del Maule',
    'UNIVERSIDAD CATOLICA DEL MAULE': 'Región del Maule',
    'UNIVERSIDAD DE ANTOFAGASTA': 'Región de Antofagasta',
    'UNIVERSIDAD CATÓLICA DEL NORTE': 'Región de Antofagasta',
    'UNIVERSIDAD CATOLICA DEL NORTE': 'Región de Antofagasta',
    'UNIVERSIDAD DE ATACAMA': 'Región de Atacama',
    'UNIVERSIDAD DE ACONCAGUA': 'Región de Valparaíso',
    'UNIVERSIDAD DE LA SERENA': 'Región de Coquimbo',
    'UNIVERSIDAD DE TARAPACA': 'Región de Arica y Parinacota',
    'UNIVERSIDAD ARTURO PRAT': 'Región de Tarapacá',
}

# ── PASO 5: CARGAR CSV ────────────────────────────────────────────────
print("Cargando CSV de ingenierías...")
try:
    _df_ing = pd.read_csv('csv_dashboard/todas_las_ingenierias_chile.csv', sep=';', encoding='utf-8')
except UnicodeDecodeError:
    _df_ing = pd.read_csv('csv_dashboard/todas_las_ingenierias_chile.csv', sep=';', encoding='utf-16')
_df_ing.columns = [c.strip().lstrip('\ufeff') for c in _df_ing.columns]

def _pct_float(s):
    return pd.to_numeric(
        s.astype(str).str.replace('%','',regex=False).str.replace(',','.',regex=False).replace('s/i', np.nan),
        errors='coerce'
    )

_df_ing['Retención de 1er año']     = _pct_float(_df_ing['Retención de 1er año'])
_df_ing['Empleabilidad al 1er año']  = _pct_float(_df_ing['Empleabilidad al 1er año'])
_df_ing['Empleabilidad al 2º Año']   = _pct_float(_df_ing['Empleabilidad al 2º Año'])
_df_ing['Sueldo_Num']                = _df_ing['Ingreso promedio al 4° año'].apply(_sueldo_a_num)
_df_ing['Region'] = _df_ing['Institución'].str.upper().str.strip().map(
    {k.upper(): v for k, v in _INST_A_REGION.items()}
)
print(f"  Filas: {len(_df_ing)} | Sin región: {_df_ing['Region'].isna().sum()}")

# ── PASO 6: KPIs POR REGIÓN ───────────────────────────────────────────
def _agg_kpis():
    rows = []
    for region, grp in _df_ing.groupby('Region'):
        g_emp = grp.dropna(subset=['Empleabilidad al 1er año'])
        g_ret = grp.dropna(subset=['Retención de 1er año'])
        g_sue = grp.dropna(subset=['Sueldo_Num'])
        rows.append({
            'Region':            region,
            'Empleabilidad_1año': grp['Empleabilidad al 1er año'].mean(),
            'Retencion':          grp['Retención de 1er año'].mean(),
            'Sueldo_4año':        grp['Sueldo_Num'].mean(),
            'Mejor_Carrera_Emp':  g_emp.loc[g_emp['Empleabilidad al 1er año'].idxmax(),'Carrera'] if not g_emp.empty else 'N/D',
            'Mejor_Emp_Val':      g_emp['Empleabilidad al 1er año'].max() if not g_emp.empty else np.nan,
            'Mejor_Carrera_Ret':  g_ret.loc[g_ret['Retención de 1er año'].idxmax(),'Carrera'] if not g_ret.empty else 'N/D',
            'Mejor_Ret_Val':      g_ret['Retención de 1er año'].max() if not g_ret.empty else np.nan,
            'Mejor_Carrera_Sue':  g_sue.loc[g_sue['Sueldo_Num'].idxmax(),'Carrera'] if not g_sue.empty else 'N/D',
            'Mejor_Sue_Val':      g_sue['Sueldo_Num'].max() if not g_sue.empty else np.nan,
            'N_Carreras':         grp['Carrera'].nunique(),
            'N_Inst':             grp['Institución'].nunique(),
        })
    return pd.DataFrame(rows)

_df_kpi = _agg_kpis()

# ── PASO 7: JOIN shapefile + KPIs (con _norm que elimina guiones) ──────
_gdf_raw['_key'] = _gdf_raw[_col_reg].apply(_norm)
_df_kpi['_key']  = _df_kpi['Region'].apply(_norm)

print(f"\n  Claves shapefile: {sorted(_gdf_raw['_key'].unique())}")
print(f"  Claves CSV:       {sorted(_df_kpi['_key'].unique())}")

_gdf_merged = _gdf_raw.merge(_df_kpi, on='_key', how='left').reset_index(drop=True)
print(f"\n  ✅ Regiones con datos: {_gdf_merged['Empleabilidad_1año'].notna().sum()}/{len(_gdf_merged)}")

# ── PASO 8: FUNCIÓN GENERADORA DE MAPA ────────────────────────────────
def _crear_mapa(col_z, titulo, colorscale, es_sueldo=False,
                col_carrera=None, col_mejor_val=None, etiqueta=''):

    geojson  = json.loads(_gdf_merged.to_json())
    ids_str  = [str(i) for i in _gdf_merged.index]   # strings '0','1','2'...
    valores  = _gdf_merged[col_z].tolist()

    hover_texts = []
    for _, row in _gdf_merged.iterrows():
        nombre = row.get(_col_reg, 'Región')
        val    = row[col_z]
        if pd.isna(val):
            hover_texts.append(f"<b>{nombre}</b><br><i>Sin datos</i>")
            continue
        val_str = f"${int(val):,}".replace(',','.') if es_sueldo else f"{val:.1f}%"
        txt = f"<b>{nombre}</b><br>{titulo}: <b>{val_str}</b>"
        if col_carrera and pd.notna(row.get(col_carrera)):
            mv   = row.get(col_mejor_val, np.nan)
            mv_s = (f"${int(mv):,}".replace(',','.') if es_sueldo else f"{mv:.1f}%") if pd.notna(mv) else ''
            txt += f"<br><br>🏆 {etiqueta}:<br><i>{row[col_carrera]}</i>"
            if mv_s: txt += f"  ({mv_s})"
        if pd.notna(row.get('N_Carreras')):
            txt += f"<br><br>📚 Carreras: {int(row['N_Carreras'])} | 🏛 Inst.: {int(row['N_Inst'])}"
        hover_texts.append(txt)

    vals_ok = [v for v in valores if v is not None and not (isinstance(v,float) and np.isnan(v))]

    fig = go.Figure(go.Choropleth(
        geojson      = geojson,
        locations    = ids_str,
        z            = valores,
        featureidkey = 'id',
        colorscale   = colorscale,
        colorbar     = dict(
            title      = dict(text=titulo, font=dict(size=10)),
            thickness  = 14, len=0.65, x=1.0,
            tickformat = '$,.0f' if es_sueldo else '.1f',
        ),
        hovertext        = hover_texts,
        hoverinfo        = 'text',
        marker_line_color= 'white',
        marker_line_width= 0.8,
        zmin = float(min(vals_ok)) if vals_ok else 0,
        zmax = float(max(vals_ok)) if vals_ok else 100,
    ))

    fig.update_geos(
        visible          = False,
        projection_type  = 'mercator',
        lataxis_range    = [-56, -17],
        lonaxis_range    = [-76, -64],
    )
    fig.update_layout(
        title         = dict(text=titulo, x=0.5, font=dict(size=12, color='#2c3e50')),
        height        = 420,
        width         = 555,
        margin        = dict(l=0, r=85, t=35, b=5),
        paper_bgcolor = 'rgba(0,0,0,0)',
    )
    return fig

# ── PASO 9: MOTOR DE RENDERIZADO Y CONEXIÓN AL DASHBOARD ──────────────

def _renderizar_mapa(change):
    with area_imagen_mapa:
        clear_output(wait=False)
        opcion = selector_mapas.value

        if opcion == '---' or opcion is None:
            # FIX 1: Texto ajustado sin flexbox para que no se corte hacia arriba
            display(widgets.HTML("""
                <div style='text-align:center; padding-top:90px;'>
                    <div style='font-size:3em;'>🗺️</div>
                    <h4 style='color:#bbb; margin:12px 0 8px 0;'>Elige un mapa en el panel</h4>
                    <p style='font-size:0.85em; color:#ccc;'>
                        Pasa el cursor sobre las regiones<br>para ver información detallada
                    </p>
                </div>
            """))
            return

        # BLOQUE DE DETECCIÓN DE ERRORES INTERNOS
        try:
            if 'Empleabilidad' in opcion:
                fig = _crear_mapa('Empleabilidad_1año', 'Empleabilidad 1er Año (%)', 'RdYlGn',
                                  col_carrera='Mejor_Carrera_Emp', col_mejor_val='Mejor_Emp_Val',
                                  etiqueta='Mayor empleabilidad')
            elif 'Retenci' in opcion:
                fig = _crear_mapa('Retencion', 'Retención 1er Año (%)', 'Blues',
                                  col_carrera='Mejor_Carrera_Ret', col_mejor_val='Mejor_Ret_Val',
                                  etiqueta='Mayor retención')
            else:
                fig = _crear_mapa('Sueldo_4año', 'Sueldo Promedio 4° Año', 'YlOrRd', es_sueldo=True,
                                  col_carrera='Mejor_Carrera_Sue', col_mejor_val='Mejor_Sue_Val',
                                  etiqueta='Mayor sueldo')

            # FIX 2: Renderizamos convirtiendo la figura a HTML para evitar bloqueos del FigureWidget
            html_str = fig.to_html(include_plotlyjs='cdn', full_html=False)
            display(HTML(html_str))

        except Exception as e:
            import traceback
            error_html = f"""
            <div style='padding: 20px; color: #721c24; background-color: #f8d7da; border: 1px solid #f5c6cb; border-radius: 5px; height: 100%; overflow-y: auto; text-align: left;'>
                <h4 style='margin-top: 0;'>🚨 Error interno al procesar el mapa</h4>
                <pre style='background: white; padding: 10px; font-size: 0.8em; border-radius: 4px; overflow-x: auto;'>{traceback.format_exc()}</pre>
            </div>
            """
            display(widgets.HTML(error_html))

# FIX 3: Conexión segura al widget. Eliminamos unobserve_all() para no romper el botón.
try:
    selector_mapas.unobserve(_renderizar_mapa, names='value')
except Exception:
    pass

selector_mapas.observe(_renderizar_mapa, names='value')

print("✅ Celda de mapas actualizada: Bug de renderizado y texto cortado solucionados.")

Cargando shapefile...
  Columna región: 'Region'
  Regiones tras filtrar bbox: 17
Cargando CSV de ingenierías...
  Filas: 306 | Sin región: 0

  Claves shapefile: ['region de antofagasta', 'region de arica y parinacota', 'region de atacama', 'region de aysen del gralibanez del campo', 'region de coquimbo', 'region de la araucania', 'region de los lagos', 'region de los rios', 'region de magallanes y antartica chilena', 'region de nuble', 'region de tarapaca', 'region de valparaiso', 'region del biobio', 'region del libertador bernardo ohiggins', 'region del maule', 'region metropolitana de santiago', 'zona sin demarcar']
  Claves CSV:       ['region de antofagasta', 'region de arica y parinacota', 'region de atacama', 'region de coquimbo', 'region de la araucania', 'region de los lagos', 'region de los rios', 'region de tarapaca', 'region de valparaiso', 'region del biobio', 'region del maule', 'region metropolitana de santiago']

  ✅ Regiones con datos: 12/17
✅ Celda de mapas actualiz

In [19]:
# %%
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 3 — Lógica y Renderizado con Datos Reales SIES    ║
# ╚══════════════════════════════════════════════════════════╝
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

ANCHO_FIJO = '980px'
ALTO_FIJO = '580px'

# 1. Enrutador general de pestañas superiores
def alternar_pestanas(change):
    with contenedor_cuerpo:
        clear_output(wait=True)
        pestana_activa = change['new'] if change else tabs_navegacion.value
        
        if pestana_activa == 'KPIs':
            display(layout_tab_kpis)
            actualizar_kpi_cards(None)
        elif pestana_activa == 'GRÁFICOS':
            display(layout_tab_graficos)
            actualizar_imagen_grafico(None)
        elif pestana_activa == 'MAPAS':
            display(layout_tab_mapas)
            _renderizar_mapa(None)
        elif pestana_activa == 'ML (RANDOM FOREST PESOS)':
            display(layout_tab_ml)

tabs_navegacion.observe(alternar_pestanas, names='value')


# 2. MOTOR DE KPIs REALES: Cruza tus selectores con la nueva base de datos chile.csv
def actualizar_kpi_cards(change):
    with area_kpi_cards:
        clear_output(wait=True)
        institucion = selector_kpi_institucion.value
        carrera = selector_kpi_carrera.value
        
        if institucion == '---' or carrera == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'>"
                                 "<h4>[ Selecciona una institucion y una carrera para desplegar las metricas KPI ]</h4></div>"))
        else:
            # Segmentación tolerante para buscar en la base de datos nacional
            m_inst = institucion.lower().replace('pontificia ', '').split(' de ')[0]
            
            dic_carreras = {
                'Ingenieria Civil Informatica': 'informática|computación|software',
                'Ingenieria Civil Industrial': 'industrial',
                'Ingenieria Civil en Obras Civiles': 'obras civiles|civil',
                'Ingenieria Comercial': 'comercial'
            }
            m_carr = dic_carreras.get(carrera, 'invalid')
            
            # Filtramos directamente sobre el nuevo caché nacional
            df_res = cache_kpi_real[
                (cache_kpi_real['Institución'].str.lower().str.contains(m_inst, na=False)) &
                (cache_kpi_real['Carrera'].str.lower().str.contains(m_carr, na=False))
            ]
            
            if not df_res.empty:
                # Extraemos las métricas reales del archivo
                fila = df_res.iloc[0]
                
                v_ret = f"{fila['Retención de 1er año']:.1f}%" if not np.isnan(fila['Retención de 1er año']) else "--%"
                v_emp = f"{fila['Empleabilidad al 2º Año']:.1f}%" if not np.isnan(fila['Empleabilidad al 2º Año']) else "--%"
                v_dur = f"{fila['Duración Real (semestres)']:.1f} sem" if not np.isnan(fila['Duración Real (semestres)']) else "-- sem"
                
                # Formateamos las lucas del ingreso promedio (Ej: $1.450.000)
                ingreso_num = fila['Ingreso promedio al 4° año']
                v_ingreso = f"${int(ingreso_num):,}".replace(',', '.') if not np.isnan(ingreso_num) else "No disponible"
            else:
                v_ret, v_emp, v_dur, v_ingreso = "--%", "--%", "-- sem", "--"
                
            html_content = f"""
            <div style='font-family: sans-serif; padding: 5px; height:100%;'>
                <h4 style='color: #2c3e50; margin-top: 0; margin-bottom: 5px;'>Datos Oficiales Mifuturo: {institucion}</h4>
                <h5 style='color: #555; margin-top: 0; margin-bottom: 15px; font-weight: normal;'>Programa: <b>{carrera}</b></h5>
                
                <div style='display: flex; justify-content: space-between; gap: 10px; margin-bottom: 15px;'>
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5bc0de; padding: 10px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Retención</div>
                        <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_ret}</div>
                        <div style='font-size: 0.72em; color: #999; margin-top: 2px;'>Pasan a 2.° año</div>
                    </div>
                        
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5cb85c; padding: 10px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Empleabilidad</div>
                        <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_emp}</div>
                        <div style='font-size: 0.72em; color: #999; margin-top: 2px;'>Al 2.° año de egreso</div>
                    </div>
                    
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #f0ad4e; padding: 10px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Duración Real</div>
                        <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_dur}</div>
                        <div style='font-size: 0.72em; color: #999; margin-top: 2px;'>Tiempo real de egreso</div>
                    </div>
                </div>
                
                <div style='background-color: #eef9f0; border: 1px solid #c3e6cb; border-left: 6px solid #28a745; padding: 12px; border-radius: 4px; margin-bottom: 15px; text-align: center;'>
                    <div style='font-size: 0.85em; color: #155724; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;'>💰 Sueldo Promedio Estimado al 4° Año de Titulación</div>
                    <div style='font-size: 1.8em; font-weight: bold; color: #1e7e34; margin-top: 5px;'>{v_ingreso}</div>
                </div>
                
                <div style='background-color: #fff; border: 1px solid #e3e3e3; padding: 10px; border-radius: 4px;'>
                    <p style='font-size: 0.82em; color: #666; line-height: 1.3; margin-bottom: 0;'>
                        <b>Nota de Transparencia:</b> Los índices de Retención, Duración y Empleabilidad corresponden a los registros oficiales auditados por el Servicio de Información de Educación Superior (SIES).
                    </p>
                </div>
            </div>
            """
            display(widgets.HTML(html_content))

selector_kpi_institucion.observe(actualizar_kpi_cards, names='value')
selector_kpi_carrera.observe(actualizar_kpi_cards, names='value')





# 4. MOTOR DE GRÁFICOS INTERACTIVOS (Plotly Express)
def actualizar_imagen_grafico(change):
    with area_imagen_grafico:
        clear_output(wait=True)
        opcion_seleccionada = selector_graficos.value
        
        if opcion_seleccionada == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'>"
                                 "<h4>[ Selecciona un reporte del panel izquierdo para desplegar el grafico ]</h4></div>"))
        else:
            if opcion_seleccionada == 'Brecha de Genero en Matematicas (M1 vs M2)':
                df_g = cache_graficos.groupby('Año')[['Matrícula primer año hombres', 'Matrícula primer año mujeres']].sum().reset_index()
                df_g['Porcentaje Mujeres (%)'] = (df_g['Matrícula primer año mujeres'] / (df_g['Matrícula primer año hombres'] + df_g['Matrícula primer año mujeres'])) * 100
                fig = px.line(df_g, x='Año', y='Porcentaje Mujeres (%)', markers=True)
                fig.update_yaxes(range=[0, 50])
            elif opcion_seleccionada == 'Distribucion por Rama Educacional (HC vs TP)':
                df_g = cache_graficos.groupby('Año')['Matrícula Total'].sum().reset_index()
                fig = px.line(df_g, x='Año', y='Matrícula Total', markers=True, labels={'Matrícula Total': 'Comunidad Total Estudiantes'})
            elif opcion_seleccionada == 'Evolucion del Puntaje por Dependencia':
                df_g = cache_graficos.groupby('Año')['Valor de arancel'].median().reset_index()
                fig = px.line(df_g, x='Año', y='Valor de arancel', markers=True, labels={'Valor de arancel': 'Arancel Mediano ($)'})
                fig.update_layout(yaxis_tickformat="$")
            else:
                display(widgets.HTML("<div style='padding:20px;'><h4>Reporte No Soportado</h4></div>"))
                return

            fig.update_layout(height=380, width=590, margin=dict(l=10, r=10, t=30, b=10), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
            fig.show()

selector_graficos.observe(actualizar_imagen_grafico, names='value')


# === 5. LOGIN Y SEGURIDAD COMPARTIDA ===
txt_usuario = widgets.Text(description='Usuario:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
txt_password = widgets.Password(description='Clave:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
btn_login = widgets.Button(description='Autenticar', button_style='primary', icon='lock', layout=widgets.Layout(margin='20px 0px 5px 0px', width='280px'))
html_feedback = widgets.HTML(value="")

formulario_interno = widgets.VBox([
    widgets.HTML("<h3 style='text-align: center; font-family: sans-serif; color: #333; margin-top:0;'>SISTEMA DE ACCESO</h3><hr style='width: 100%; border: 0; border-top: 1px solid #ccc;'>"),
    txt_usuario, txt_password, btn_login, html_feedback
], layout=widgets.Layout(width='360px', padding='25px', border='1px solid #ccc', bg_color='#ffffff', align_items='center', border_radius='4px'))

cuadro_login = widgets.VBox([formulario_interno], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', bg_color='#f4f4f4', justify_content='center', align_items='center'))
dashboard_final = widgets.VBox([estilos_css, tabs_navegacion, linea_separadora, contenedor_cuerpo], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', bg_color='#f4f4f4', padding='20px'))

def validar_credenciales(b):
    if txt_usuario.value == 'admin' and txt_password.value == 'admin':
        with lienzo_maestro:
            clear_output()
            display(dashboard_final)
            alternar_pestanas(None)
    else:
        txt_password.value = ""
        html_feedback.value = "<div style='color: #d9534f; font-weight: bold; text-align: center; margin-top: 12px; font-family: sans-serif;'>Error: Credenciales Incorrectas</div>"

btn_login.on_click(validar_credenciales)
lienzo_maestro = widgets.Output()

print("Celda 3: Motores reactivos cruzados con la base nacional ejecutados con éxito.")

Celda 3: Motores reactivos cruzados con la base nacional ejecutados con éxito.


In [20]:
# %%
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 4 — Inicialización y Lanzamiento en Pantalla       ║
# ╚══════════════════════════════════════════════════════════╝

# 1. Desplegamos el nodo de salida raíz (el lienzo maestro) en Jupyter
display(lienzo_maestro)

# 2. Forzamos el renderizado inicial de la caja de login dentro del lienzo
with lienzo_maestro:
    clear_output()
    display(cuadro_login)

Output()